# Agentic RAG: Router_retriever System

This notebook builds an agentic retrieval-augmented generation system centered on a router agent and a retriever agent: the router agent classifies an incoming question to decide the best source of information, then the retriever agent answers the question by pulling context from a local PDF, performing a live web search, or falling back to a direct LLM call when neither retrieval path is needed.

In [1]:
%pip install -q -r requirements.txt

Note: you may need to restart the kernel to use updated packages.


In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "your-openai-api-key")
OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-4o-mini")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY", "your-tavily-api-key")
EMBEDDING_PROVIDER = os.getenv("EMBEDDING_PROVIDER", "onnx")

os.environ.setdefault("OPENAI_API_KEY", OPENAI_API_KEY)
os.environ.setdefault("OPENAI_MODEL", OPENAI_MODEL)
os.environ.setdefault("TAVILY_API_KEY", TAVILY_API_KEY)
os.environ.setdefault("EMBEDDING_PROVIDER", EMBEDDING_PROVIDER)

if OPENAI_API_KEY == "your-openai-api-key":
    raise ValueError(
        "OPENAI_API_KEY is still set to its placeholder value. "
        "Fill in a real key in your .env file."
    )
if TAVILY_API_KEY == "your-tavily-api-key":
    raise ValueError(
        "TAVILY_API_KEY is still set to its placeholder value. "
        "Fill in a real key in your .env file."
    )

In [3]:
from crewai import LLM

llm = LLM(
    model=OPENAI_MODEL,
    api_key=OPENAI_API_KEY,
    temperature=0,
    max_tokens=512,
)

In [4]:
from crewai_tools import PDFSearchTool


def _notebook_dir():
    vsc_path = globals().get("__vsc_ipynb_file__")
    if vsc_path:
        return os.path.dirname(os.path.abspath(vsc_path))
    return os.path.abspath(os.getcwd())


PDF_PATH = os.path.abspath(
    os.path.join(_notebook_dir(), "data", "pdfs", "trasformer_research_paper-dataset.pdf")
)

if not os.path.isfile(PDF_PATH):
    raise FileNotFoundError(
        f"Expected PDF not found at {PDF_PATH}. "
        "Place the source PDF in data/pdfs/ before running this cell."
    )

pdf_search_tool = PDFSearchTool(
    pdf=PDF_PATH,
    config=dict(
        embedder=dict(
            provider=EMBEDDING_PROVIDER,
        ),
        vectordb=dict(
            provider="chromadb",
            config=dict(
                batch_size=1,
            ),
        ),
    ),
)

In [5]:
import requests
from crewai.tools import BaseTool


class TavilySearchTool(BaseTool):
    name: str = "Web_Search"
    description: str = (
        "Searches the live web via Tavily for up-to-date information. "
        "Input should be a natural-language search query."
    )

    def _run(self, query: str) -> str:
        try:
            response = requests.post(
                "https://api.tavily.com/search",
                json={
                    "api_key": TAVILY_API_KEY,
                    "query": query,
                    "max_results": 3,
                },
                timeout=15,
            )
        except requests.exceptions.RequestException as exc:
            return f"Web search request failed: {exc}"

        if not response.ok:
            try:
                error_detail = response.json().get("error", response.text)
            except ValueError:
                error_detail = response.text
            return f"Web search unavailable ({response.status_code}): {error_detail}"

        results = response.json().get("results", [])
        if not results:
            return "Web search returned no results."

        return "\n".join(
            f"{result.get('title', 'Untitled')} - {result.get('url', '')}"
            for result in results
        )


web_search_tool = TavilySearchTool()

In [6]:
from datetime import datetime

TRACE_LOG = []


def log_event(step, details):
    entry = {
        "timestamp": datetime.now().isoformat(timespec="seconds"),
        "step": step,
        "details": details,
    }
    TRACE_LOG.append(entry)
    print(f"[{entry['timestamp']}] {step}: {details}")
    return entry


PDF_KEYWORDS = (
    "pdf", "document", "paper", "report", "section", "chapter",
    "according to", "in the document",
)
WEB_KEYWORDS = (
    "latest", "current", "recent", "today", "now", "news", "update", "online",
)


def route_question(question):
    lowered = question.lower()
    if any(keyword in lowered for keyword in PDF_KEYWORDS):
        route = "pdf"
    elif any(keyword in lowered for keyword in WEB_KEYWORDS):
        route = "web"
    else:
        route = "direct"
    log_event("route_question", {"question": question, "route": route})
    return route


def _trim_response(text, limit=1800):
    if text is None:
        return text
    if len(text) <= limit:
        return text
    return text[:limit] + "[truncated]"


def _is_web_error(result):
    if not isinstance(result, str):
        return False
    return result.startswith("Web search unavailable") or result.startswith(
        "Web search request failed"
    )

In [7]:
def _call_llm(prompt):
    return llm.call(prompt)


def retrieve_answer(route, question):
    log_event("dispatch", {"route": route, "question": question})

    if route == "pdf":
        raw_result = pdf_search_tool.run(query=question)
        trimmed_result = _trim_response(str(raw_result))
        log_event("pdf_retrieval", {"result": trimmed_result})

        refine_prompt = (
            "You are answering a question using only the PDF excerpts below. "
            "Write one short paragraph, then up to 3 bullet points of supporting "
            "evidence quoted or paraphrased from the excerpts.\n\n"
            f"Question: {question}\n\n"
            f"PDF excerpts:\n{trimmed_result}"
        )
        try:
            result = _call_llm(refine_prompt)
            log_event(
                "pdf_refinement", {"answer_preview": _trim_response(str(result), 200)}
            )
        except Exception as exc:
            log_event("pdf_refinement_failed", {"error": str(exc)})
            result = trimmed_result

    elif route == "web":
        raw_result = web_search_tool.run(query=question)
        if _is_web_error(raw_result):
            log_event("web_search_failed", {"result": raw_result})
            fallback_prompt = (
                "Real-time web search was unavailable, so current information "
                "could not be verified. Answer the question from your own "
                "knowledge, and explicitly tell the user that real-time "
                "verification was not available and the answer may be out of "
                "date.\n\n"
                f"Question: {question}"
            )
            result = _call_llm(fallback_prompt)
            log_event(
                "web_fallback_direct",
                {"answer_preview": _trim_response(str(result), 200)},
            )
        else:
            log_event("web_search", {"result": _trim_response(str(raw_result))})
            result = raw_result

    else:
        result = _call_llm(question)
        log_event("direct_llm", {"answer_preview": _trim_response(str(result), 200)})

    log_event("tool_completed", {"route": route, "char_count": len(str(result))})
    return result

In [9]:
test_questions = [
    "According to the paper, what does the document say about the transformer architecture?",
    "What is the latest AI news today?",
    "What is the capital of France?",
]

for question in test_questions:
    route = route_question(question)
    answer = retrieve_answer(route, question)
    print(f"\nQ: {question}\nRoute: {route}\nA: {answer}\n{'-' * 60}")

print("\n=== TRACE LOG ===")
for entry in TRACE_LOG:
    print(entry)

[2026-09-09T13:43:30] route_question: {'question': 'According to the paper, what does the document say about the transformer architecture?', 'route': 'pdf'}
[2026-09-09T13:43:30] dispatch: {'route': 'pdf', 'question': 'According to the paper, what does the document say about the transformer architecture?'}


[2026-09-09T13:43:30] pdf_retrieval: {'result': 'Relevant Content:\n\nto averaging attention-weighted positions, an effect we counteract with Multi-Head Attention as\n\ndescribed in section 3.2.\n\nSelf-attention, sometimes called intra-attention is an attention mechanism relating different positions\n\nof a single sequence in order to compute a representation of the sequence. Self-attention has been\n\nused successfully in a variety of tasks including reading comprehension, abstractive summarization,\n\ntextual entailment and learning task-independent sentence representations [4, 27, 28, 22].\n\nEnd-to-end memory networks are based on a recurrent attention mechanism instead of sequence-\n\naligned recurrence and have been shown to perform well on simple-language question answering and\n\nlanguage modeling tasks [34].\n\nTo the best of our knowledge, however, the Transformer is the first transduction model relying\n\nentirely on self-attention to compute representations of its input an

[2026-09-09T13:43:32] pdf_refinement: {'answer_preview': 'The document describes the transformer architecture as a groundbreaking model that utilizes self-attention mechanisms to process input and output sequences without relying on traditional recurrent neu[truncated]'}
[2026-09-09T13:43:32] tool_completed: {'route': 'pdf', 'char_count': 1026}

Q: According to the paper, what does the document say about the transformer architecture?
Route: pdf
A: The document describes the transformer architecture as a groundbreaking model that utilizes self-attention mechanisms to process input and output sequences without relying on traditional recurrent neural networks (RNNs) or convolutional layers. This innovative approach allows the transformer to compute representations of sequences more effectively, making it suitable for various tasks such as reading comprehension and summarization.

- The transformer is noted as the first transduction model that relies entirely on self-attention for computin

[2026-09-09T13:43:32] direct_llm: {'answer_preview': 'The capital of France is Paris.'}
[2026-09-09T13:43:32] tool_completed: {'route': 'direct', 'char_count': 31}

Q: What is the capital of France?
Route: direct
A: The capital of France is Paris.
------------------------------------------------------------

=== TRACE LOG ===
{'timestamp': '2026-09-09T13:43:30', 'step': 'route_question', 'details': {'question': 'According to the paper, what does the document say about the transformer architecture?', 'route': 'pdf'}}
{'timestamp': '2026-09-09T13:43:30', 'step': 'dispatch', 'details': {'route': 'pdf', 'question': 'According to the paper, what does the document say about the transformer architecture?'}}
{'timestamp': '2026-09-09T13:43:30', 'step': 'pdf_retrieval', 'details': {'result': 'Relevant Content:\n\nto averaging attention-weighted positions, an effect we counteract with Multi-Head Attention as\n\ndescribed in section 3.2.\n\nSelf-attention, sometimes called intra-attention is an a

In [8]:
import json

from crewai import Agent


class RouteDecisionTool(BaseTool):
    name: str = "Route_Question"
    description: str = (
        "Classifies a user question into one of three retrieval routes: "
        "'pdf', 'web', or 'direct'. Input should be the raw question text."
    )

    def _run(self, question: str) -> str:
        return route_question(question)


class AnswerRetrievalTool(BaseTool):
    name: str = "Retrieve_Answer"
    description: str = (
        "Fetches a grounded answer for a question given a chosen route. "
        "Input must be a JSON string with 'route' and 'question' keys, e.g. "
        '{"route": "pdf", "question": "..."}.'
    )

    def _run(self, payload: str) -> str:
        try:
            data = json.loads(payload)
        except (json.JSONDecodeError, TypeError) as exc:
            return f"Invalid payload, expected JSON with 'route' and 'question': {exc}"

        route = data.get("route")
        question = data.get("question")
        if not route or not question:
            return "Invalid payload: both 'route' and 'question' keys are required."

        return retrieve_answer(route, question)


route_tool = RouteDecisionTool()
answer_tool = AnswerRetrievalTool()

router_agent = Agent(
    role="Router Agent",
    goal="Choose the best retrieval route for a user question",
    backstory=(
        "You are responsible for reading each incoming question and deciding "
        "whether it should be answered from the PDF, a live web search, or "
        "directly from the LLM's own knowledge."
    ),
    tools=[route_tool],
    llm=llm,
    allow_delegation=False,
)

retriever_agent = Agent(
    role="Retriever Agent",
    goal="Use the chosen route to fetch a grounded answer",
    backstory=(
        "You are responsible for taking the route chosen by the Router Agent "
        "and retrieving a grounded, evidence-backed answer to the user's "
        "question."
    ),
    tools=[answer_tool],
    llm=llm,
    allow_delegation=False,
)

In [9]:
from crewai import Task, Crew, Process


def print_trace_summary():
    print("=== TRACE SUMMARY ===")
    for i, entry in enumerate(TRACE_LOG, start=1):
        print(f"{i}. [{entry['timestamp']}] {entry['step']}: {entry['details']}")


async def run_agentic_rag(question):
    TRACE_LOG.clear()
    log_event("question_received", {"question": question})

    routing_task = Task(
        description=(
            "Classify the following user question into exactly one retrieval "
            "route using the Route_Question tool, then answer with only that "
            "single word: 'pdf', 'web', or 'direct'. No other text.\n\n"
            f"Question: {question}"
        ),
        expected_output="A single word: pdf, web, or direct.",
        agent=router_agent,
    )
    routing_crew = Crew(
        agents=[router_agent],
        tasks=[routing_task],
        process=Process.sequential,
    )
    routing_result = await routing_crew.kickoff_async(inputs={"question": question})

    raw_route = str(routing_result).strip().lower()
    if "pdf" in raw_route:
        route = "pdf"
    elif "web" in raw_route:
        route = "web"
    elif "direct" in raw_route:
        route = "direct"
    else:
        route = "direct"
    log_event("route_resolved", {"raw_output": raw_route, "route": route})

    payload = json.dumps({"route": route, "question": question})
    retrieval_task = Task(
        description=(
            "Use the Retrieve_Answer tool with the following JSON payload as "
            "input to fetch a grounded answer, then return that answer "
            f"verbatim.\n\nPayload: {payload}"
        ),
        expected_output="The grounded answer text returned by the tool.",
        agent=retriever_agent,
    )
    retrieval_crew = Crew(
        agents=[retriever_agent],
        tasks=[retrieval_task],
        process=Process.sequential,
    )
    retrieval_result = await retrieval_crew.kickoff_async(inputs={"payload": payload})

    final_answer = str(retrieval_result)
    log_event(
        "final_answer",
        {"length": len(final_answer), "preview": _trim_response(final_answer, 200)},
    )

    print_trace_summary()
    print("\n=== FINAL ANSWER ===")
    print(final_answer)

    return final_answer

In [10]:
async def run_demo_suite():
    demo_cases = [
        ("PDF", "pdf", "According to the PDF document, summarize the main topic."),
        (
            "Web",
            "web",
            "What are the latest developments in agentic AI orchestration today?",
        ),
        ("Direct", "direct", "Explain retrieval-augmented generation in simple terms."),
    ]

    results = []
    for label, expected_route, question in demo_cases:
        print(f"\n{'=' * 20} DEMO CASE: {label} {'=' * 20}")
        answer = await run_agentic_rag(question)

        resolved_route = None
        for entry in TRACE_LOG:
            if entry["step"] == "route_resolved":
                resolved_route = entry["details"].get("route")
                break

        reasons = []
        route_ok = resolved_route == expected_route
        if not route_ok:
            reasons.append(f"expected route '{expected_route}', got '{resolved_route}'")

        length_ok = len(answer.strip()) > 40
        if not length_ok:
            reasons.append(f"answer too short ({len(answer.strip())} chars)")

        web_ok = True
        if expected_route == "web":
            web_ok = not _is_web_error(answer)
            if not web_ok:
                reasons.append("answer looks like a disguised web-search-failure message")

        passed = route_ok and length_ok and web_ok
        status = "PASS" if passed else "FAIL"
        reason_text = "; ".join(reasons) if reasons else "all checks passed"
        print(f"[{status}] {label}: {reason_text}")
        results.append(passed)

    passed_count = sum(results)
    print(f"\n{passed_count}/{len(results)} passed")


await run_demo_suite()


==================== DEMO CASE: PDF ====================
[2026-09-09T13:48:01] question_received: {'question': 'According to the PDF document, summarize the main topic.'}


[2026-09-09T13:48:04] route_question: {'question': 'According to the PDF document, summarize the main topic.', 'route': 'pdf'}


[2026-09-09T13:48:05] route_resolved: {'raw_output': 'pdf', 'route': 'pdf'}


╭─────────────────────────────────────────── Tracing Preference Saved ────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing has been disabled.                                                                               │
│                                                                                                                 │
│  Your preference has been saved. Future Crew/Flow executions will not collect traces.                           │
│                                                                                                                 │
│  To enable tracing later, do any one of these:                                                                  │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[2026-09-09T13:48:05] dispatch: {'route': 'pdf', 'question': 'According to the PDF document, summarize the main topic.'}


...

[2026-09-09T13:48:18] route_resolved: {'raw_output': 'direct', 'route': 'direct'}

[2026-09-09T13:48:19] dispatch: {'route': 'direct', 'question': 'Explain retrieval-augmented generation in simple terms.'}


[2026-09-09T13:48:21] direct_llm: {'answer_preview': "Retrieval-Augmented Generation (RAG) is a method used in natural language processing that combines two key techniques: retrieving information and generating text.\n\nHere's how it works in simple terms:[truncated]"}
[2026-09-09T13:48:21] tool_completed: {'route': 'direct', 'char_count': 1007}


[2026-09-09T13:48:23] final_answer: {'length': 1007, 'preview': "Retrieval-Augmented Generation (RAG) is a method used in natural language processing that combines two key techniques: retrieving information and generating text.\n\nHere's how it works in simple terms:[truncated]"}
=== TRACE SUMMARY ===
1. [2026-09-09T13:48:16] question_received: {'question': 'Explain retrieval-augmented generation in simple terms.'}
2. [2026-09-09T13:48:17] route_question: {'question': 'Explain retrieval-augmented generation in simple terms.', 'route': 'direct'}
3. [2026-09-09T13:48:18] route_resolved: {'raw_output': 'direct', 'route': 'direct'}
4. [2026-09-09T13:48:19] dispatch: {'route': 'direct', 'question': 'Explain retrieval-augmented generation in simple terms.'}
5. [2026-09-09T13:48:21] direct_llm: {'answer_preview': "Retrieval-Augmented Generation (RAG) is a method used in natural language processing that combines two key techniques: retrieving information and generating text.\n\nHere's how it w

╭─────────────────────────────────────────── Tracing Preference Saved ────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing has been disabled.                                                                               │
│                                                                                                                 │
│  Your preference has been saved. Future Crew/Flow executions will not collect traces.                           │
│                                                                                                                 │
│  To enable tracing later, do any one of these:                                                                  │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯